In [8]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator


In [2]:
load_dotenv()
model = ChatOpenAI(model='gpt-4o-mini')


In [19]:
class EvaluationSchema(BaseModel):
    score: int = Field(description="Score between 0 and 10", ge=0, le=10)
    feedback: str = Field(description="Feedback on the content")

In [21]:
structured_model = model.with_structured_output(EvaluationSchema)

In [5]:
essay = """Artificial Intelligence (AI) and machine learning are two terms that have become increasingly prevalent in today's technology landscape. From virtual assistants to self-driving cars, AI and machine learning are revolutionizing the way we live and work. In this blog post, we will delve into the history, applications, challenges, and the future of AI and machine learning.\n\nI. Introduction\nAI refers to the simulation of human intelligence processes by machines, while machine learning is a subset of AI that allows machines to learn and improve from experience without being explicitly programmed. These technologies are crucial in today's world as they are driving innovation and transforming industries such as healthcare, finance, marketing, and more.\n\nII. History of AI and Machine Learning\nThe origins of AI can be traced back to the 1950s, with key developments such as the creation of the first neural network in 1958. Over the years, AI has evolved, leading to breakthroughs like IBM's Deep Blue defeating world chess champion Garry Kasparov in 1997. These advancements have had a profound impact on society, changing the way we live and work.\n\nIII. Applications of AI and Machine Learning\nAI and machine learning are being used in a wide range of industries, from diagnosing medical conditions to predicting stock market trends. Some successful applications include Google's search engine algorithms and Amazon's recommendation system. The potential for further development and innovation in the field is vast, with opportunities for advancement in various sectors.\n\nIV. Challenges and Ethical Considerations\nDespite the numerous benefits of AI and machine learning, there are challenges and ethical considerations that need to be addressed. Issues such as data privacy and security, bias in algorithms, and potential job displacement are concerns that need to be carefully managed to ensure the responsible development and deployment of AI technologies.\n\nV. The Future of AI and Machine Learning\nEmerging trends in AI and machine learning include the rise of autonomous vehicles, the use of AI in cybersecurity, and the development of more sophisticated natural language processing systems. With opportunities for further development and innovation abundant, the future of AI and machine learning looks promising.\n\nVI. Conclusion\nIn conclusion, AI and machine learning have the potential to revolutionize the way we live and work. It is important for us to understand the history, applications, challenges, and the future of these technologies to fully grasp their significance. I encourage readers to further explore the topic of AI and machine learning to stay informed and engaged in this exciting field."""

In [9]:
class EssayFeedbackState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float
    

In [11]:
def evaluate_language(state: EssayFeedbackState) :
    prompt = f"Evaluate the language quality of the following essay and give a score between 0 and 10: {state['essay']}"
    response = structured_model.invoke(prompt)

    return {
        "language_feedback": response.feedback,
        "individual_scores": [response.score]
    }



In [12]:
def evaluate_analysis(state: EssayFeedbackState) :
    prompt = f"Evaluate the depth of analysis of the following essay and give a score between 0 and 10: {state['essay']}"
    response = structured_model.invoke(prompt)

    return {
        "analysis_feedback": response.feedback,
        "individual_scores": [response.score]
    }
    

In [14]:
def evaluate_clarity(state: EssayFeedbackState) :
    prompt = f"Evaluate the clarity of thought of the following essay and give a score between 0 and 10: {state['essay']}"
    response = structured_model.invoke(prompt)

    return {
        "clarity_feedback": response.feedback,
        "individual_scores": [response.score]
    }

In [15]:
def final_evaluation(state: EssayFeedbackState) :
    #summary
    prompt = f"based on the following feedback, give a summary of the essay: {state['language_feedback']}, {state['analysis_feedback']}, {state['clarity_feedback']}"
    overall_feedback = model.invoke(prompt).content

    #average calculation
    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])

    return {
        "overall_feedback": overall_feedback,
        "avg_score": avg_score
    }

In [17]:
graph = StateGraph(EssayFeedbackState)

graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_clarity", evaluate_clarity)
graph.add_node("final_evaluation", final_evaluation)

graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_clarity")

graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_analysis", "final_evaluation")
graph.add_edge("evaluate_clarity", "final_evaluation")

graph.add_edge("final_evaluation", END)

workflow = graph.compile()

In [22]:
initial_state = {
    "essay": essay
}

final_result = workflow.invoke(initial_state)
print(final_result)

{'essay': "Artificial Intelligence (AI) and machine learning are two terms that have become increasingly prevalent in today's technology landscape. From virtual assistants to self-driving cars, AI and machine learning are revolutionizing the way we live and work. In this blog post, we will delve into the history, applications, challenges, and the future of AI and machine learning.\n\nI. Introduction\nAI refers to the simulation of human intelligence processes by machines, while machine learning is a subset of AI that allows machines to learn and improve from experience without being explicitly programmed. These technologies are crucial in today's world as they are driving innovation and transforming industries such as healthcare, finance, marketing, and more.\n\nII. History of AI and Machine Learning\nThe origins of AI can be traced back to the 1950s, with key developments such as the creation of the first neural network in 1958. Over the years, AI has evolved, leading to breakthroughs